In [1]:
# Create SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession siap. Versi Spark:", spark.version)

26/10/08 13:00:39 WARN Utils: Your hostname, yan resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/08 13:00:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 13:00:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/08 13:00:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


In [2]:
# A. EXTRACT
base = "hdfs://localhost:9000/user/yan/tugas7/raw"

df_transaksi = spark.read.csv(f"{base}/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json(f"{base}/tugas7_cabang.json")
df_menu = spark.read.csv(f"{base}/tugas7_menu.csv", header=True, inferSchema=True)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Transaksi: 4000 | Cabang: 3 | Menu: 15


In [3]:
# B. TRANSFORM — Join & Kolom Turunan
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                          .join(df_menu, on="menu_id", how="inner") \
                          .withColumn("total_penjualan", col("qty") * col("harga"))

df_gabungan.show(5)
print("Jumlah baris gabungan:", df_gabungan.count())

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [4]:
# C. TRANSFORM — Window Function
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu", "kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2_menu = ringkasan_menu_cabang \
    .withColumn("peringkat", row_number().over(window_cabang)) \
    .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show(truncate=False)

+-------------------------+---------+--------------+---------------+---------+
|nama_cabang              |nama_menu|kategori_menu |total_penjualan|peringkat|
+-------------------------+---------+--------------+---------------+---------+
|Kopi Nusantara Magelang  |Menu-1   |Makanan Ringan|8505000        |1        |
|Kopi Nusantara Magelang  |Menu-12  |Makanan Ringan|8085000        |2        |
|Kopi Nusantara Semarang  |Menu-13  |Makanan Ringan|8750000        |1        |
|Kopi Nusantara Semarang  |Menu-9   |Makanan Ringan|8715000        |2        |
|Kopi Nusantara Yogyakarta|Menu-12  |Makanan Ringan|9275000        |1        |
|Kopi Nusantara Yogyakarta|Menu-15  |Non-Kopi      |8160000        |2        |
+-------------------------+---------+--------------+---------------+---------+



In [5]:
# D. TRANSFORM — Spark SQL
df_gabungan.createOrReplaceTempView("gabungan")

penjualan_barista = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")
penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [6]:
# E. LOAD
!hdfs dfs -mkdir -p /user/yan/tugas7/processed

path_laporan = "hdfs://localhost:9000/user/yan/tugas7/processed/laporan"

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(path_laporan)

!hdfs dfs -ls /user/yan/tugas7/processed/laporan

df_verifikasi = spark.read.parquet(path_laporan)
print("Jumlah baris setelah dibaca ulang:", df_verifikasi.count())

Found 4 items
-rw-r--r--   3 yan supergroup          0 2026-10-08 13:01 /user/yan/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - yan supergroup          0 2026-10-08 13:00 /user/yan/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - yan supergroup          0 2026-10-08 13:01 /user/yan/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - yan supergroup          0 2026-10-08 13:01 /user/yan/tugas7/processed/laporan/kategori_menu=Non-Kopi
Jumlah baris setelah dibaca ulang: 4000


In [7]:
# F. Ringkasan Eksekutif & Rekomendasi
ringkasan_eksekutif = df_verifikasi.groupBy("nama_cabang").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif.show(truncate=False)

+-------------------------+---------------+----------------+-------------------------+
|nama_cabang              |total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|84645000       |1305            |64862.06896551724        |
+-------------------------+---------------+----------------+-------------------------+



Cabang terbaik adalah Kopi Nusantara Semarang. Total penjualannya sekitar Rp90,5 juta dari 1.393 transaksi. Urutan berikutnya adalah Magelang (sekitar Rp87,1 juta) dan Yogyakarta (sekitar Rp84,6 juta). Semarang menang karena jumlah transaksinya paling banyak. Magelang punya rata-rata belanja per transaksi paling tinggi, yaitu sekitar Rp66.900. Selisih cabang terbaik dan terlemah hanya sekitar 7%, jadi ketiga cabang hampir setara.

Cabang terlemah adalah Yogyakarta. Dua menu terlarisnya di sana adalah Menu-12 (Makanan Ringan, sekitar Rp9,3 juta) dan Menu-15 (Non-Kopi, sekitar Rp8,2 juta). Saranku, promosikan Menu-15 lebih banyak, misalnya lewat paket hemat bersama Menu-12. Alasannya, Menu-15 sudah laku di Yogyakarta dan harganya Rp30.000, jadi bisa menaikkan rata-rata belanja di cabang yang paling rendah. Stok Menu-12 tetap dijaga supaya pelanggan mau membeli paketnya.

In [8]:
# Close SparkSession
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
